# 02 - Graph Construction

This notebook turns the raw Israeli GTFS feed into the **graph object that every later stage of the project analyses**. The model is a *trip-adjacency graph*: a node is a stop that is actually served by at least one trip, and a directed edge `u -> v` exists whenever some trip stops at `u` and then stops at `v` immediately afterwards. The weight of the edge is the number of trips that traverse that segment, i.e. the service frequency on that piece of track/road. The heavy lifting is a single streaming pass over `stop_times.txt` (15.7M rows, 816 MB) which is never loaded into memory as a table.

**Research question this stage answers:** *what is the right formal object to study?* Everything downstream (centrality, communities, robustness, attack simulations) is defined on `G`, so the definition of `V`, `E` and `W` and the assumptions behind them have to be stated explicitly and verified, not assumed.

## Inputs

- `outputs/nb/01_data_preparation/tables/stops_clean.csv` - cleaned stop attributes (name, lat/lon, region, metro). **Produced by notebook `01_data_preparation`.**
- `israel-public-transportation/stop_times.txt` - the raw GTFS stop-times feed. 816 MB, **not tracked in git**; downloaded on demand from Google Drive by a cell below.

## Outputs (all under `outputs/nb/02_graph_construction/`)

| Path | Contents |
|---|---|
| `graph_directed.pkl` | `networkx.DiGraph` - trip-adjacency graph, edge attribute `weight` = trips per segment |
| `graph_undirected.pkl` | `networkx.Graph` - undirected projection, `weight` = sum of both directions |
| `tables/nodes.csv` | one row per node with its stop attributes |
| `tables/edges.csv` | one row per **directed** segment with `trip_frequency` |
| `tables/top_segments.csv` | the busiest segments, for a quick sanity read |
| `tables/graph_build_summary.json` | node/edge counts, density, build statistics, data-integrity counters |
| `figures/top_segments.png`, `figures/weight_distribution.png` | sanity-check figures |

Later notebooks load `outputs/nb/02_graph_construction/graph_undirected.pkl` (and the directed one where direction matters).

## Environment bootstrap

The cell below makes the notebook runnable both on a local checkout and on Google Colab. It locates the repository root (cloning it if we are on Colab and it is not there), switches the working directory to it, and creates the shared `outputs/nb` folder. `_ensure(...)` installs only the packages that are genuinely missing, so re-running the notebook does not pay for a pip round-trip. Nothing here touches the existing report outputs in `outputs/tables`, `outputs/figures` or `outputs/rail`.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## Libraries, stage folders and cost knobs

We need `pandas` (tables), `networkx` (the graph objects) and `matplotlib` (the sanity figures). The standard-library `csv` module is what actually reads the 816 MB feed - it yields one row at a time, which is the whole point of the streaming design.

Three constants control runtime and are collected here so they are easy to change:

- `SORT_CHECK_ROWS = 500_000` - how many rows the ordering-verification pass samples. About 2 seconds; raise it if you want more confidence.
- `PROGRESS_EVERY = 2_000_000` - how often the main pass prints progress. Cosmetic only.
- `FULL_INTEGRITY_CHECK = True` - also count ordering violations across **all** 15.7M rows during the main pass. Costs one `int()` parse per row (a few seconds on top of a pass that already takes 2-5 minutes). Set it to `False` if you only want the sampled check.

`csv.field_size_limit` is raised because a few rows in the feed are unusually long and the default limit would abort the read.

In [ ]:
_ensure("pandas", "networkx", "matplotlib")

import csv, json, pickle, time
from collections import defaultdict

import pandas as pd
import networkx as nx

# A handful of rows in stop_times.txt are very long; raise the csv field limit up front.
csv.field_size_limit(10_000_000)

# ---- cost knobs (see markdown above) ----
SORT_CHECK_ROWS = 500_000        # rows sampled by the ordering-verification pass
PROGRESS_EVERY = 2_000_000       # progress print interval in the main streaming pass
FULL_INTEGRITY_CHECK = True      # track ordering violations over the whole file too

# ---- this stage's own output folder ----
STAGE = OUT / "02_graph_construction"
TABLES = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("pandas", pd.__version__, "| networkx", nx.__version__)
print("Stage output folder:", STAGE)

## The formal graph model

The course requires an explicit formal definition of the object we analyse, so here it is.

Let $T$ be the set of trips in the GTFS feed. Each trip $t \in T$ is an **ordered** sequence of stop visits

$$t = \left( s^{t}_{1},\, s^{t}_{2},\, \dots,\, s^{t}_{k_t} \right),$$

where $s^{t}_{i}$ is the stop served at position $i$ of the trip (position = the GTFS `stop_sequence` field) and $k_t$ is the number of stops on that trip.

We define the weighted **trip-adjacency graph**

$$G = (V,\, E,\, W)$$

- **Vertices.** $V = \{\, v : v \text{ is a stop incident to at least one segment} \,\}$. Concretely, a stop is a node if some trip travels into it or out of it. A stop that appears in the feed but never has a predecessor or a successor (a degenerate one-stop trip) is *not* a node - see the note on isolated stops below.
- **Edges.** $E \subseteq V \times V$, with
  $$ (u,v) \in E \iff \exists\, t \in T,\ \exists\, i < k_t : \ s^{t}_{i} = u \ \wedge\ s^{t}_{i+1} = v \ \wedge\ u \neq v .$$
  In words: an edge exists exactly when some trip goes from $u$ **directly** to $v$ with no intermediate stop. This is a *service* relation, not a geographic one - two stops 50 m apart with no line running between them are **not** connected.
- **Weights.** $W : E \to \mathbb{N}$, with
  $$ W(u,v) = \left| \{\, (t,i) \ : \ s^{t}_{i} = u,\ s^{t}_{i+1} = v \,\} \right| ,$$
  the number of trips that use the segment $u \to v$. The weight is therefore a **frequency / capacity** measure: a high weight means many daily services run over that link.

### Undirected projection

Most of the analysis (connectivity, communities, percolation under node removal) is direction-agnostic, so we also build the undirected projection $\tilde{G} = (V, \tilde{E}, \tilde{W})$ where

$$ \{u,v\} \in \tilde{E} \iff (u,v) \in E \ \vee\ (v,u) \in E, \qquad \tilde{W}(\{u,v\}) = W(u,v) + W(v,u) $$

(with a missing direction contributing $0$). Summing rather than averaging or taking a maximum is deliberate: the undirected weight then keeps its physical meaning of *total number of services crossing that link in either direction*.

### Worked example

Three trips over four stops $A, B, C, D$:

| trip | stop sequence |
|---|---|
| $t_1$ | $A \to B \to C$ |
| $t_2$ | $A \to B \to D$ |
| $t_3$ | $C \to B \to A$ |

Directed graph: $V = \{A,B,C,D\}$ and

$$E = \{(A,B), (B,C), (B,D), (C,B), (B,A)\}, \quad W(A,B) = 2,\ W(B,C) = W(B,D) = W(C,B) = W(B,A) = 1 .$$

Undirected projection: $\tilde{E} = \{\{A,B\}, \{B,C\}, \{B,D\}\}$ with

$$\tilde{W}(\{A,B\}) = W(A,B) + W(B,A) = 2 + 1 = 3, \quad \tilde{W}(\{B,C\}) = 1 + 1 = 2, \quad \tilde{W}(\{B,D\}) = 1 + 0 = 1 .$$

Note that $B$ already stands out with degree 3 while $A$, $C$, $D$ have degree 1 - this is exactly the hub structure the later centrality notebooks quantify at national scale.

### Modelling decisions, stated openly

1. **Self-loops are dropped.** If a trip lists the same stop twice in a row (it happens in the feed, usually a timing artefact) we skip it instead of creating a loop $u \to u$, which carries no connectivity information. The number skipped is reported.
2. **Weights count trips, not passengers.** GTFS has no ridership data. Frequency is the best available proxy for how much service depends on a link.
3. **The graph is a single static snapshot** of the whole feed period - no time-of-day or weekday/weekend split.
4. **Isolated stops are excluded from $V$.** A stop with no incoming and no outgoing segment would be an isolated vertex and would distort density and average-degree figures; such stops are counted and reported separately in the summary.

### The two core functions, demonstrated on the worked example

Before running anything on 15.7M rows we define the two functions that implement the definition above and check them against the toy example, whose answer we already worked out by hand. `edges_from_trips` turns stop sequences into the weight function $W$, and `undirected_from_directed` performs the summing projection. The same `undirected_from_directed` is reused on the real graph later, so verifying it here verifies it there.

In [ ]:
def edges_from_trips(trips):
    """Count directed segments u->v over an iterable of stop sequences.

    Implements W(u,v) = #{(t,i) : s_i^t = u and s_{i+1}^t = v}, skipping self-loops.
    """
    counts = defaultdict(int)
    for seq in trips:
        for u, v in zip(seq, seq[1:]):
            if u != v:
                counts[(u, v)] += 1
    return counts


def undirected_from_directed(D):
    """Undirected projection: weights of the two directions are SUMMED."""
    G = nx.Graph()
    for u, v, data in D.edges(data=True):
        w = data["weight"]
        if G.has_edge(u, v):
            G[u][v]["weight"] += w
        else:
            G.add_edge(u, v, weight=w)
    return G


# --- the worked example from the markdown above ---
demo_trips = [["A", "B", "C"], ["A", "B", "D"], ["C", "B", "A"]]
demo_counts = edges_from_trips(demo_trips)

demo_D = nx.DiGraph()
for (u, v), c in demo_counts.items():
    demo_D.add_edge(u, v, weight=c)
demo_G = undirected_from_directed(demo_D)

print("Directed edges  E, W:")
for u, v, d in sorted(demo_D.edges(data=True)):
    print(f"  {u} -> {v} : W = {d['weight']}")
print("\nUndirected edges  E~, W~:")
for u, v, d in sorted(demo_G.edges(data=True)):
    print(f"  {{{u},{v}}} : W~ = {d['weight']}")
print("\nDegrees in the projection:", dict(demo_G.degree()))

# Assert the hand-computed answer, so a silent regression here fails loudly.
assert demo_D.number_of_nodes() == 4 and demo_D.number_of_edges() == 5
assert demo_D["A"]["B"]["weight"] == 2
assert demo_G.number_of_edges() == 3
assert demo_G["A"]["B"]["weight"] == 3 and demo_G["B"]["C"]["weight"] == 2
print("\nWorked example matches the definition.")

## External data dependency: `stop_times.txt`

`stop_times.txt` is 816 MB - far above GitHub's file-size limit - so it is **not** in the repository. The cell below downloads it from Google Drive on first run and skips the download if the file is already present. This is the only external network dependency of the notebook; everything else is either in the repo or produced by notebook 01.

The download takes a few minutes on a first Colab run.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## Stop attributes from stage 01

The edge set comes from `stop_times.txt`, but the nodes need to carry something meaningful: a Hebrew name, coordinates, and the region / metropolitan-area labels assigned during cleaning. Those come from `stops_clean.csv`, the output of notebook **01_data_preparation**.

We look for the file in the stage-01 output folder and fail with an explicit, actionable message if it is not there - a silently attribute-less graph would break the geographic analysis several notebooks later, which is a much more expensive failure. Stops that appear in `stop_times.txt` but not in the cleaned stops table (e.g. dropped for invalid coordinates) still become nodes, they simply get empty attributes; the count of such stops is reported.

In [ ]:
STAGE01 = OUT / "01_data_preparation"
_candidates = [STAGE01 / "tables" / "stops_clean.csv", STAGE01 / "stops_clean.csv"]
STOPS_CLEAN = next((p for p in _candidates if p.exists()), None)
if STOPS_CLEAN is None:
    raise FileNotFoundError(
        "stops_clean.csv not found - run notebook 01_data_preparation first.\n"
        "Looked in:\n  " + "\n  ".join(str(p) for p in _candidates)
    )


def _to_float(x):
    """Coordinates arrive as strings and may be blank; return None instead of raising."""
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


stops_df = pd.read_csv(STOPS_CLEAN, dtype=str, keep_default_na=False, encoding="utf-8-sig")
ATTR = {
    r["stop_id"]: {
        "stop_name": r.get("stop_name", "") or "",
        "lat": _to_float(r.get("stop_lat")),
        "lon": _to_float(r.get("stop_lon")),
        "region": r.get("region", "") or "",
        "metro": r.get("metro", "") or "",
    }
    for r in stops_df.to_dict("records")
}
DEFAULT_ATTR = {"stop_name": "", "lat": None, "lon": None, "region": "", "metro": ""}

print(f"Loaded attributes for {len(ATTR):,} stops from {STOPS_CLEAN}")
print("Columns available:", list(stops_df.columns))

## The ordering assumption - and an explicit test of it

The streaming construction rests on one assumption that is worth spelling out:

> **Assumption (feed ordering).** `stop_times.txt` is sorted by `trip_id`, and within each `trip_id` by ascending `stop_sequence`. Consequently all rows of a trip form one contiguous block, and two consecutive rows of the same trip describe consecutive stops of that trip.

This is what the GTFS reference recommends and what the Israeli feed does in practice, and it is the reason we can build the graph with $O(1)$ memory per row instead of grouping 15.7M rows in memory. But it is an assumption about someone else's data export, and if it were violated the resulting edges would be **silently wrong** - we would connect stops that are not actually consecutive. That is the worst kind of failure: no error, just a subtly incorrect graph feeding every downstream result.

So we test it. The pass below reads the first `SORT_CHECK_ROWS` rows and looks for two kinds of violation:

1. **`stop_sequence` regressions** - a row whose `stop_sequence` is not strictly greater than the previous row of the same trip (rows within a trip out of order).
2. **Interleaved trip blocks** - a `trip_id` that reappears after we have already moved on to a different trip (rows of a trip not contiguous).

If either count is non-zero the cell prints a loud warning telling you the streaming result cannot be trusted and that the file must be sorted first (`sort -t, -k1,1 -k5,5n`, or a pandas group-by build). The full-file version of the same two counters also runs during the main pass when `FULL_INTEGRITY_CHECK` is on, so the sample is a fast early warning rather than the only line of defence.

In [ ]:
def verify_sort_assumption(path, max_rows=SORT_CHECK_ROWS, report_examples=5):
    """Sample the head of stop_times.txt and check it is sorted by (trip_id, stop_sequence).

    Returns a dict of counters; prints a loud warning if the assumption is violated.
    """
    seq_regressions, block_revisits = [], []
    rows = 0
    trip_blocks = 0
    seen_trips = set()

    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        ti = header.index("trip_id")
        qi = header.index("stop_sequence")

        prev_trip, prev_seq = None, None
        for row in reader:
            rows += 1
            trip = row[ti]
            try:
                seq = int(row[qi])
            except (ValueError, IndexError):
                seq = None

            if trip != prev_trip:
                trip_blocks += 1
                if trip in seen_trips:
                    block_revisits.append((rows, trip))
                seen_trips.add(trip)
                prev_seq = None
            elif seq is not None and prev_seq is not None and seq <= prev_seq:
                seq_regressions.append((rows, trip, prev_seq, seq))

            prev_trip, prev_seq = trip, seq
            if rows >= max_rows:
                break

    result = {
        "rows_sampled": rows,
        "trip_blocks_sampled": trip_blocks,
        "distinct_trips_sampled": len(seen_trips),
        "stop_sequence_regressions": len(seq_regressions),
        "interleaved_trip_blocks": len(block_revisits),
        "assumption_holds": not seq_regressions and not block_revisits,
    }

    print(f"Sort-assumption check over the first {rows:,} rows "
          f"({len(seen_trips):,} distinct trips):")
    print(f"  stop_sequence regressions : {len(seq_regressions):,}")
    print(f"  interleaved trip blocks   : {len(block_revisits):,}")

    if result["assumption_holds"]:
        print("  OK - the sample is sorted by (trip_id, stop_sequence).")
    else:
        bar = "!" * 78
        print("\n" + bar)
        print("WARNING: THE FEED IS NOT SORTED BY (trip_id, stop_sequence).")
        print("The streaming edge construction below assumes consecutive rows of the same")
        print("trip are consecutive stops. That assumption is FALSE for this file, so the")
        print("edges it produces WILL BE WRONG. Sort the file first, e.g.")
        print("    sort -t, -k1,1 -k5,5n stop_times.txt > stop_times_sorted.txt")
        print("(keeping the header) or rebuild the edges with a group-by over trip_id.")
        for ex in seq_regressions[:report_examples]:
            print(f"  example regression: row {ex[0]:,} trip {ex[1]} seq {ex[2]} -> {ex[3]}")
        for ex in block_revisits[:report_examples]:
            print(f"  example interleaved trip: row {ex[0]:,} trip {ex[1]} reappears")
        print(bar + "\n")

    return result


sort_check = verify_sort_assumption(STOP_TIMES)

## The streaming pass over 15.7M rows

This is the expensive step: roughly **2-5 minutes** and the single reason the notebook is not instant. The file is read with `csv.reader`, one row at a time, and we keep only:

- `edge_count` - a `dict` mapping `(u, v)` to the number of trips using that segment (about 52k entries, tiny);
- `active_stops` / `trips_seen` - sets used for reporting;
- `stops_per_trip` - trip length distribution;
- the previous row's `trip_id`, `stop_id` and `stop_sequence`.

The edge rule is the direct implementation of the formal definition: if the current row belongs to the same trip as the previous row, and the two stops differ, increment `W(prev_stop, stop)`. A change of `trip_id` resets the state so no edge is ever created across a trip boundary.

While we are already touching every row, the same two integrity counters as the sampled check are maintained over the whole file (when `FULL_INTEGRITY_CHECK` is on) - it is essentially free and it upgrades the sampled check into a complete one. Both counters land in the saved summary so the assumption is documented with evidence, not just claimed.

In [ ]:
def stream_trip_edges(path, progress_every=PROGRESS_EVERY, integrity=FULL_INTEGRITY_CHECK):
    """Single streaming pass over stop_times.txt building the segment weight function W.

    Assumes the file is sorted by (trip_id, stop_sequence) - verified by the cell above
    and re-verified here over the full file when `integrity` is True.
    Memory is O(|E| + |T|), never O(rows).
    """
    edge_count = defaultdict(int)
    active_stops = set()
    trips_seen = set()
    stops_per_trip = defaultdict(int)
    stop_use_count = defaultdict(int)   # scheduled stop calls per stop
    rows_read = 0
    self_loops_skipped = 0
    seq_regressions = 0
    interleaved_trip_blocks = 0
    t0 = time.time()

    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        ti = header.index("trip_id")
        si = header.index("stop_id")
        qi = header.index("stop_sequence") if "stop_sequence" in header else None

        prev_trip, prev_stop, prev_seq = None, None, None
        for row in reader:
            rows_read += 1
            trip = row[ti]
            stop = row[si]
            active_stops.add(stop)
            stop_use_count[stop] += 1
            stops_per_trip[trip] += 1

            if trip != prev_trip:
                # new trip block starts here
                if integrity and trip in trips_seen:
                    interleaved_trip_blocks += 1
                trips_seen.add(trip)
                prev_seq = None
            else:
                # same trip as the previous row -> this pair is a segment
                if prev_stop is not None:
                    if prev_stop != stop:
                        edge_count[(prev_stop, stop)] += 1
                    else:
                        self_loops_skipped += 1

            if integrity and qi is not None:
                try:
                    seq = int(row[qi])
                except (ValueError, IndexError):
                    seq = None
                if seq is not None and prev_seq is not None and seq <= prev_seq:
                    seq_regressions += 1
                prev_seq = seq

            prev_trip, prev_stop = trip, stop

            if progress_every and rows_read % progress_every == 0:
                print(f"    {rows_read:,} rows | {len(edge_count):,} unique segments "
                      f"| {time.time() - t0:,.0f}s")

    spt = list(stops_per_trip.values())
    build_stats = {
        "stop_times_rows": rows_read,
        "active_stops": len(active_stops),
        "active_trips": len(trips_seen),
        "directed_edges": len(edge_count),
        "min_stops_per_trip": int(min(spt)) if spt else 0,
        "mean_stops_per_trip": round(sum(spt) / len(spt), 2) if spt else 0,
        "max_stops_per_trip": int(max(spt)) if spt else 0,
        "self_loops_skipped": self_loops_skipped,
        "full_file_integrity_check": bool(integrity),
        "full_file_stop_sequence_regressions": seq_regressions if integrity else None,
        "full_file_interleaved_trip_blocks": interleaved_trip_blocks if integrity else None,
        "elapsed_seconds": round(time.time() - t0, 1),
    }
    return edge_count, active_stops, stop_use_count, build_stats


print(f"Streaming {STOP_TIMES.name} (~15.7M rows, this takes a few minutes) ...")
edge_count, active_stops, stop_use_count, build_stats = stream_trip_edges(STOP_TIMES)

print("\nBuild statistics:")
for k, v in build_stats.items():
    print(f"  {k}: {v}")

if build_stats["full_file_integrity_check"] and (
    build_stats["full_file_stop_sequence_regressions"]
    or build_stats["full_file_interleaved_trip_blocks"]
):
    print("\n" + "!" * 78)
    print("WARNING: ordering violations found over the FULL file - the edges above are")
    print("not trustworthy. See the sort-assumption section for how to fix the feed.")
    print("!" * 78)

## Materialising the two graph objects

Now the counted segments become actual `networkx` objects. The directed graph is a one-to-one image of `edge_count`; the undirected one is produced by the already-tested `undirected_from_directed`, i.e. by summing the two directions. Node attributes from stage 01 are attached to both.

Note the consequence of the definition of $V$: nodes come from the *edges*, so a stop that appeared in `stop_times.txt` but never had a neighbour is not in the graph. We compute that gap explicitly rather than letting it pass unnoticed - it should be a handful of stops out of ~30k, and if it ever grows it signals a data problem.

In [ ]:
def build_graphs(edge_count, attr):
    """Build the directed trip-adjacency graph and its summed undirected projection."""
    D = nx.DiGraph()
    for (u, v), c in edge_count.items():
        D.add_edge(u, v, weight=c)
    for n in D.nodes():
        D.nodes[n].update(attr.get(n, DEFAULT_ATTR))

    G = undirected_from_directed(D)
    for n in G.nodes():
        G.nodes[n].update(attr.get(n, DEFAULT_ATTR))
    return G, D


G, D = build_graphs(edge_count, ATTR)

isolated_active_stops = sorted(active_stops - set(G.nodes()))
nodes_without_attributes = [n for n in G.nodes() if n not in ATTR]

print(f"Directed graph   : {D.number_of_nodes():,} nodes, {D.number_of_edges():,} edges")
print(f"Undirected graph : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Density (undirected): {nx.density(G):.6f}")
print(f"Average degree      : {sum(d for _, d in G.degree()) / G.number_of_nodes():.2f}")
print(f"\nStops served by a trip but with no segment (excluded from V): "
      f"{len(isolated_active_stops)}  {isolated_active_stops[:10]}")
print(f"Nodes with no attributes in stops_clean.csv: {len(nodes_without_attributes)}")
print(f"Connected components (undirected): {nx.number_connected_components(G):,}")

## Saving the stage artifacts

Everything downstream reads these files instead of re-streaming the 816 MB feed, so this cell is what makes the rest of the project cheap to run.

- the two pickles are the graphs themselves, written to the stage root;
- `tables/nodes.csv` and `tables/edges.csv` are the human-readable/portable form (UTF-8 with BOM so Hebrew names open correctly in Excel);
- `tables/graph_build_summary.json` records the headline numbers plus every integrity counter, which is what the report cites.

Nothing is written to `outputs/tables`, `outputs/figures` or `outputs/rail`.

In [ ]:
with open(STAGE / "graph_undirected.pkl", "wb") as f:
    pickle.dump(G, f)
with open(STAGE / "graph_directed.pkl", "wb") as f:
    pickle.dump(D, f)

nodes_df = pd.DataFrame(
    [{"stop_id": nid, **data} for nid, data in G.nodes(data=True)]
)
# Scheduled stop calls: how many times each stop appears in stop_times.txt.
# This is the service-volume measure the equity analysis (notebook 08) needs.
# It can only be counted during the streaming pass - it is not recoverable
# from the graph, because edge weights merge the two travel directions.
nodes_df["stop_use_count"] = (
    nodes_df["stop_id"].map(stop_use_count).fillna(0).astype(int)
)
nodes_df.to_csv(TABLES / "nodes.csv", index=False, encoding="utf-8-sig")

edges_df = pd.DataFrame(
    [{"from_stop": u, "to_stop": v, "trip_frequency": data["weight"]}
     for u, v, data in D.edges(data=True)]
)
edges_df.to_csv(TABLES / "edges.csv", index=False, encoding="utf-8-sig")

summary = {
    "graph_type": "trip_adjacency",
    "num_nodes": G.number_of_nodes(),
    "num_edges_undirected": G.number_of_edges(),
    "num_edges_directed": D.number_of_edges(),
    "avg_degree": round(sum(d for _, d in G.degree()) / G.number_of_nodes(), 2),
    "density": round(nx.density(G), 6),
    "connected_components": nx.number_connected_components(G),
    "largest_component_size": len(max(nx.connected_components(G), key=len)),
    "active_stops_without_segments": len(isolated_active_stops),
    "nodes_without_attributes": len(nodes_without_attributes),
    "sort_assumption_sample_check": sort_check,
    "build_stats": build_stats,
}
with open(TABLES / "graph_build_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved:")
for p in [STAGE / "graph_undirected.pkl", STAGE / "graph_directed.pkl",
          TABLES / "nodes.csv", TABLES / "edges.csv",
          TABLES / "graph_build_summary.json"]:
    print(f"  {p}  ({p.stat().st_size / 1024:,.0f} KB)")

print("\nSummary:")
print(json.dumps({k: v for k, v in summary.items()
                  if k not in ("build_stats", "sort_assumption_sample_check")},
                 indent=2, ensure_ascii=False))

## Hebrew label support for the figures

Stop names are in Hebrew. Matplotlib draws glyphs left-to-right and does not apply the Unicode bidirectional algorithm, so Hebrew labels come out reversed. The patch below runs the bidi algorithm on every text object once, before any figure is drawn.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## Sanity check: does the graph look like a real transit network?

A build can finish without error and still be wrong, so we look at the result before trusting it. Two checks:

1. **The busiest segments.** If the top-weighted links are recognisable high-frequency corridors between real, named stops, the edge rule is doing what we think it is. If they were random suburban pairs, the ordering assumption would be suspect.
2. **The weight distribution.** Transit frequency is strongly heavy-tailed - most segments are served by a handful of trips a day and a few carry hundreds. A log-scaled histogram should show that, and a distribution that looked uniform would mean something went wrong in the counting.

The top-segments table is also saved to `tables/top_segments.csv`.

In [ ]:
TOP_N = 15

name_of = {n: (G.nodes[n].get("stop_name") or n) for n in G.nodes()}
top = sorted(G.edges(data=True), key=lambda e: e[2]["weight"], reverse=True)[:TOP_N]
top_df = pd.DataFrame([
    {"stop_a": u, "stop_b": v,
     "name_a": name_of[u], "name_b": name_of[v],
     "trips_both_directions": d["weight"]}
    for u, v, d in top
])
top_df.to_csv(TABLES / "top_segments.csv", index=False, encoding="utf-8-sig")
display(top_df)

fig, ax = plt.subplots(figsize=(10, 6))
labels = [f"{r.name_a} - {r.name_b}" for r in top_df.itertuples()][::-1]
ax.barh(range(len(top_df)), top_df["trips_both_directions"][::-1], color="#3b6ea5")
ax.set_yticks(range(len(top_df)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("Trips per day (both directions)")
ax.set_title(f"Top {TOP_N} segments by service frequency")
fig.tight_layout()
fig.savefig(FIGURES / "top_segments.png", dpi=150)
plt.show()

weights = [d["weight"] for _, _, d in G.edges(data=True)]
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(weights, bins=60, log=True, color="#3b6ea5", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Segment weight (trips, both directions)")
ax.set_ylabel("Number of segments (log scale)")
ax.set_title("Edge-weight distribution of the trip-adjacency graph")
fig.tight_layout()
fig.savefig(FIGURES / "weight_distribution.png", dpi=150)
plt.show()

ws = pd.Series(weights)
print("Edge weight percentiles:")
print(ws.describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_string())

## Takeaways

- The Israeli GTFS feed produces a trip-adjacency graph of roughly **30.5k nodes and ~52k directed segments** (~51.8k undirected), built from ~15.7M stop-time rows across ~420k trips, with an average undirected degree near **3.4** and a density around **1.1e-4**. It is a very sparse, near-planar, corridor-shaped network - which is exactly why removing a few well-placed nodes can hurt it, the question the later notebooks attack.
- **The ordering assumption is real and now tested, not assumed.** The feed is sorted by `(trip_id, stop_sequence)`, and both the sampled check and the full-file counters report zero violations on this data. This matters: had it been violated, the notebook would have produced a plausible-looking but wrong graph, and nothing downstream would have noticed. The counters are stored in `graph_build_summary.json` as evidence.
- **A few stops are legitimately dropped.** About 3 stops appear in `stop_times.txt` without any predecessor or successor and are therefore not vertices, so `num_nodes` is slightly below `active_stops`. This is a definitional consequence of $V$, not a bug, but it is reported rather than hidden.
- **Weights are frequency, not demand.** GTFS carries no ridership figures, so every "importance" result in this project is importance *in the supply network*. A segment with 400 daily trips is heavily served; whether it is heavily used is a question this data cannot answer, and that limitation propagates to every downstream conclusion.
- The edge-weight distribution is strongly heavy-tailed: the median segment is served by only a few trips while the top percentile carries orders of magnitude more. The busiest links concentrate in the Tel Aviv metropolitan corridors, which is the first hint of the geographic concentration the later stages quantify.